# Lab 13: Information Retrieval for RAG

This submission includes both course notebooks and their completed exercises.

## Setup
The small NLP document collection is supplied directly in the notebooks. No external dataset or pretrained-model download is needed.

In [ ]:
%pip install -q scikit-learn

## Part 1 — Information Retrieval for RAG

## 1. What is Information Retrieval?

**Information retrieval (IR)** means finding relevant information from a collection of documents based on a user query.

**Example:** A student asks, *"How do transformers process words?"* The retrieval system searches course notes and returns the most relevant passages.

**Simple process:**

`Documents → Numerical representation → Compare with query → Rank → Top results`

## 2. Create a small document collection

Imagine these are short notes taken from an NLP course. Each string represents one document.

In [2]:
documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."]

for i, document in enumerate(documents):
    print(f"Document {i}: {document}")

Document 0: Transformers use self-attention to understand relationships between words.
Document 1: TF-IDF represents documents using word importance scores.
Document 2: Word2Vec learns dense vector representations of words.
Document 3: Named entity recognition identifies names, places, and organizations in text.
Document 4: Information retrieval finds relevant documents for a user query.
Document 5: Cosine similarity measures how similar two text vectors are.


## 3. Convert documents into TF-IDF vectors (sparse retrieval)

A computer cannot compare raw text directly using vector similarity. **TF-IDF** converts each document into a numerical vector.

- **TF** measures how frequently a word occurs in a document.
- **IDF** gives more importance to words that appear in fewer documents.

TF-IDF is a form of **sparse retrieval**: most word positions in the vectors are zero.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", document_vectors.shape)
print("First document vector:", document_vectors[0].toarray().round(2))

TF-IDF matrix shape: (6, 39)
First document vector: [[0.39 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.39 0.   0.   0.   0.   0.   0.39 0.   0.   0.
  0.   0.39 0.39 0.39 0.   0.   0.   0.   0.   0.   0.32]]


**Check:** The matrix has one row per document and one column per vocabulary word. A zero means that word has no TF-IDF weight in that document.

## 4. Search using a query

We use the **same TF-IDF vectorizer** to represent the user's query. Then we calculate **cosine similarity** between the query and every document.

- Higher similarity → more relevant according to TF-IDF.
- A similarity of 0 means the vectors do not share weighted terms.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

query = "How do transformers understand words?"
query_vector = vectorizer.transform([query])
scores = cosine_similarity(query_vector, document_vectors)[0]

for i, score in enumerate(scores):
    print(f"Document {i}: {score:.3f}")

Document 0: 0.633
Document 1: 0.000
Document 2: 0.173
Document 3: 0.000
Document 4: 0.000
Document 5: 0.000


## 5. Retrieve the top-k documents

Rather than reading every document, the system returns the **k** documents with the highest similarity scores.

In [5]:
k = 3
ranked_ids = scores.argsort()[::-1][:k]

for rank, i in enumerate(ranked_ids, start=1):
    print(f"Rank {rank} | Score: {scores[i]:.3f}")
    print(documents[i])
    print()

Rank 1 | Score: 0.633
Transformers use self-attention to understand relationships between words.

Rank 2 | Score: 0.173
Word2Vec learns dense vector representations of words.

Rank 3 | Score: 0.000
Information retrieval finds relevant documents for a user query.



## 6. Student exercises

**Exercise 1 — Change the query**  
Use `search()` to find notes about **named entity recognition**. Which document is ranked first?

**Exercise 2 — Change k**  
Search for `"document representation"` using `k=1` and then `k=3`. What changes?


### Reusable Search Function
Define `search()` here because the Part 1 exercises call it. It returns the results as well as printing them. Equal similarity scores are ordered by document index; tied ranks do not imply greater relevance.

In [6]:
def search(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    # Use document index to break equal-score ties consistently.
    top_ids = sorted(range(len(documents)), key=lambda i: (-scores[i], i))[:k]
    results = []
    for i in top_ids:
        result = {"id": i, "score": float(scores[i]), "text": documents[i]}
        results.append(result)
        print(f"Document {i} | Score: {scores[i]:.3f} | {documents[i]}")
    return results

### Exercise 1 — Search for Named Entity Recognition

In [7]:
exercise1_results = search("named entity recognition", k=2)
print("\nFirst-ranked document:", exercise1_results[0]["id"])
print(exercise1_results[0]["text"])

Document 3 | Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.

First-ranked document: 3
Named entity recognition identifies names, places, and organizations in text.


**Answer:** Document **3** (the fourth document with zero-based indexing) ranks first: “Named entity recognition identifies names, places, and organizations in text.” Its query terms overlap directly with the document.

### Exercise 2 — Change k

In [8]:
print("k = 1:")
k1_results = search("document representation", k=1)
print("\nk = 3:")
k3_results = search("document representation", k=3)
print("\nQuery terms:", vectorizer.build_analyzer()("document representation"))
print("Terms found in the fitted vocabulary:", [
    term for term in vectorizer.build_analyzer()("document representation")
    if term in vectorizer.vocabulary_
])

k = 1:
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.

k = 3:
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.
Document 1 | Score: 0.000 | TF-IDF represents documents using word importance scores.
Document 2 | Score: 0.000 | Word2Vec learns dense vector representations of words.

Query terms: ['document', 'representation']
Terms found in the fitted vocabulary: []


**Answer:** `k=1` returns one result, while `k=3` returns three results using the same scoring method. For this exact query, all similarities are **0**: the collection contains plural forms such as `documents` and `representations`, while the query uses `document` and `representation`. This vectorizer does not stem or lemmatize words, so those forms do not match. Increasing `k` therefore returns more tied zero-score documents rather than finding better matches. Part 2 demonstrates how to suppress such results.

## Summary

You built a small **information retrieval** system using TF-IDF and cosine similarity. It converts documents and queries into vectors, ranks documents by relevance, and returns the top results.

**Connection to RAG:** These retrieved documents can later be passed to a language model as supporting context for answer generation.

## Part 2 — Search Functions and Retrieval Approaches

## Setup — Run this cell first

The cell below recreates the small document collection and TF-IDF vectors used in Part 1.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

## 1. Make a simple search function

Now combine the steps so you can search the document collection using any query.

In [10]:
def search(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    # Use document index to break equal-score ties consistently.
    top_ids = sorted(range(len(documents)), key=lambda i: (-scores[i], i))[:k]
    results = []
    for i in top_ids:
        result = {"id": i, "score": float(scores[i]), "text": documents[i]}
        results.append(result)
        print(f"Document {i} | Score: {scores[i]:.3f} | {documents[i]}")
    return results

search("How can we measure similarity between text vectors?")

Document 5 | Score: 0.686 | Cosine similarity measures how similar two text vectors are.
Document 3 | Score: 0.148 | Named entity recognition identifies names, places, and organizations in text.


[{'id': 5,
  'score': 0.686385840059243,
  'text': 'Cosine similarity measures how similar two text vectors are.'},
 {'id': 3,
  'score': 0.1484991395809418,
  'text': 'Named entity recognition identifies names, places, and organizations in text.'}]

## 2. Sparse vs. dense retrieval (conceptual comparison)

| Sparse retrieval | Dense retrieval |
|---|---|
| Represents text using word-based features, e.g., TF-IDF | Represents text using learned embeddings |
| Works well when important query words appear in documents | Can identify similar meanings even when different words are used |
| Simple and lightweight | Usually requires a pretrained embedding model |

**Important limitation:** TF-IDF mainly matches words, not meanings. A query about *"automobiles"* may not retrieve a document about *"cars"* unless the words also overlap in other ways.

In modern RAG, retrieval can be sparse, dense, or a combination of both.

## 3. Does a high-ranked result always mean a good match?

The search function returns the **highest-scoring** documents, even when all the scores are zero. A high *rank* does not always mean a high *similarity score*.

For example, our collection is about NLP. What happens when we search for an unrelated topic like `"volcano eruption"`?

In [11]:
search("volcano eruption", k=2)

# Compare it with a relevant query
print("--- Relevant query ---")
search("cosine similarity", k=2)

Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.
Document 1 | Score: 0.000 | TF-IDF represents documents using word importance scores.
--- Relevant query ---
Document 5 | Score: 0.594 | Cosine similarity measures how similar two text vectors are.
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.


[{'id': 5,
  'score': 0.5937869868149507,
  'text': 'Cosine similarity measures how similar two text vectors are.'},
 {'id': 0,
  'score': 0.0,
  'text': 'Transformers use self-attention to understand relationships between words.'}]

## 4. Improve search: skip results with a score of zero

We can improve the function by printing only documents with a **positive similarity score**.

In [12]:
def search_filtered(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    ranked_ids = sorted(range(len(documents)), key=lambda i: (-scores[i], i))
    positive_ids = [i for i in ranked_ids if scores[i] > 0][:k]
    results = []
    for i in positive_ids:
        result = {"id": i, "score": float(scores[i]), "text": documents[i]}
        results.append(result)
        print(f"Document {i} | Score: {scores[i]:.3f} | {documents[i]}")
    if not results:
        print("No matching documents found.")
    return results

search_filtered("volcano eruption")
print("---")
search_filtered("cosine similarity")

No matching documents found.
---
Document 5 | Score: 0.594 | Cosine similarity measures how similar two text vectors are.


[{'id': 5,
  'score': 0.5937869868149507,
  'text': 'Cosine similarity measures how similar two text vectors are.'}]

## 5. Practice questions

1. Use `search()` to find a note about **named entity recognition**. Which document ranks first?
2. Add `"BERT is a transformer model that uses bidirectional context."` to `documents`, rebuild the TF-IDF vectors using `vectorizer.fit_transform(documents)`, and search for `"transformer models"`. Why was rebuilding necessary?

### Practice 1 — Named Entity Recognition

In [13]:
practice1_results = search("named entity recognition", k=2)
print("\nFirst-ranked document:", practice1_results[0]["id"])

Document 3 | Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.

First-ranked document: 3


**Answer:** Document **3** ranks first again. The setup recreates the same six-document collection, so the relevant query has the same best match.

### Practice 2 — Add the BERT Note and Rebuild the Index

In [14]:
print("Before adding the document:")
before_addition_results = search_filtered("transformer models", k=2)
old_vocabulary = set(vectorizer.get_feature_names_out())

new_document = "BERT is a transformer model that uses bidirectional context."
if new_document not in documents:
    documents.append(new_document)

# Refit on the complete, updated collection to rebuild its vocabulary and IDF weights.
document_vectors = vectorizer.fit_transform(documents)
print("\nUpdated document-vector matrix:", document_vectors.shape)
print("New vocabulary terms:", sorted(set(vectorizer.get_feature_names_out()) - old_vocabulary))
print("\nSearch after rebuilding:")
practice2_results = search("transformer models", k=2)
print("\nPositive-score results only:")
practice2_filtered_results = search_filtered("transformer models", k=2)

Before adding the document:
No matching documents found.

Updated document-vector matrix: (7, 45)
New vocabulary terms: ['bert', 'bidirectional', 'context', 'model', 'transformer', 'uses']

Search after rebuilding:
Document 6 | Score: 0.408 | BERT is a transformer model that uses bidirectional context.
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.

Positive-score results only:
Document 6 | Score: 0.408 | BERT is a transformer model that uses bidirectional context.


**Answer:** The new BERT note, document **6**, ranks first. Rebuilding adds a vector row for the new document, learns new vocabulary terms, and updates IDF values for the changed collection. Merely appending a string does not update the stored matrix. The fitted vectorizer must then transform queries in the same feature space as the rebuilt document vectors. Here the query matches `transformer`; `models` and `model` remain different tokens without stemming.

## Final Summary
This lab builds the retrieval component of a RAG pipeline: convert documents and a query to TF-IDF vectors, calculate cosine similarities, rank the collection, and return the top `k` results. Filtering out zero scores prevents unrelated results from appearing as matches, although a positive score alone does not prove relevance. Sparse retrieval depends on vocabulary overlap; dense retrieval uses learned embeddings and can capture semantic similarity across different wording. A complete RAG system would pass the retrieved text to a language model for answer generation, which is beyond this lab's retrieval exercises.